# 01 — Train

End-to-end training notebook. Configure one experiment in the cell below, run all cells. Output lands in `runs/<run_name>/`.

**One config per notebook execution.** Re-run the notebook with different configs to fill the headline 12-run matrix:

| algo | reward | target |
|---|---|---|
| ppo / reinforce | binding_affinity | jnk3 / egyrase |
| ppo / reinforce | multi_obj | jnk3 / egyrase |
| ppo / reinforce | qed | none |
| ppo / reinforce | sa | none |

For headless overnight runs of all 12 configs, use `python scripts/run_all.py` (when added in a follow-up session).

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import matplotlib.pyplot as plt

from lisardd.config import ExperimentConfig
from lisardd.runner import run_experiment
from lisardd.io import load_run

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## Configure this run

Edit the cell below for the (algo, reward, target) you want to train. The default reproduces the headline `reinforce_multi_jnk3` run from the camera-ready paper.

In [ ]:
cfg = ExperimentConfig(
    run_name="reinforce_multi_jnk3",
    algo="reinforce",
    reward="multi_obj",
    target="jnk3",
    seed=42,
    mode="full",
    n_epochs=100,
    batch_size=64,
    n_top=100,
    device=device,
)
print(cfg)

## Train

In [ ]:
run_dir = run_experiment(cfg)
print(f"Run saved to: {run_dir}")

## Smoke checks

Quick reward trajectory + sanity statistics. Paper-quality figures and the top-100 table live in `notebooks/02_analyze.ipynb`.

In [ ]:
art = load_run(run_dir, load_state=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(art.history['average_obj_scores'], label='avg reward')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[0].set_title(f"{cfg.run_name}"); axes[0].grid(True); axes[0].legend()

axes[1].plot(art.history['loss_actor_list'], label='actor loss')
if 'loss_critic_list' in art.history:
    axes[1].plot(art.history['loss_critic_list'], label='critic loss')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Loss')
axes[1].set_title('Training losses'); axes[1].grid(True); axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
import numpy as np

rewards_arr = np.array(art.history['average_obj_scores'])
first10 = rewards_arr[:10].mean()
last10 = rewards_arr[-10:].mean()
print(f"Mean reward, first 10 epochs: {first10:.4f}")
print(f"Mean reward, last 10 epochs:  {last10:.4f}")
print(f"Improvement: {last10 - first10:+.4f}")

valid_top = art.top100[art.top100['smiles'].notna()]
print(f"\nTop-{len(art.top100)} hits, {len(valid_top)} decoded successfully")
print(f"Top reward: {art.top100['reward'].max():.4f}")

In [ ]:
instr = art.meta.get('instrumentation', {}).get('records', [])
if instr:
    walls = [r['wall_time'] for r in instr]
    invalids = [r['n_invalid'] / r['batch_size'] for r in instr]
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax2 = ax1.twinx()
    ax1.plot(walls, color='steelblue', label='wall time / epoch (s)')
    ax2.plot(invalids, color='crimson', label='decode failure rate', alpha=0.6)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Wall time (s)', color='steelblue')
    ax2.set_ylabel('Failure rate', color='crimson')
    ax1.set_title('Per-epoch wall time vs decode failure rate (slowdown diagnostic)')
    ax1.grid(True); plt.tight_layout(); plt.show()
else:
    print('No instrumentation records found.')

## Done

Next: load this run alongside others in `notebooks/02_analyze.ipynb` for the paper-quality PPO-vs-REINFORCE comparison and paired t-test.